# 00 — Data Download

Downloads both datasets to Google Drive for all downstream Colab notebooks.

| Dataset | GEO | Contents | Role |
|---------|-----|----------|------|
| Bhaduri et al. 2020, Nature | GSE132672 | Cortical organoids — 37 organoids, 3 protocols | Organoid data |
| Zhong et al. 2018, Nature | GSE104276 | Fetal PFC — gestational weeks 8–26 | Fetal brain reference |

**Run this notebook once.** After the data is on Drive, all other notebooks load from there.

## 1. Mount Google Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## 2. Set Paths and Config

All paths are defined here. If you reorganize your Drive, only change this cell.

In [ ]:
import os

DRIVE_ROOT = '/content/drive/MyDrive/brain-organoid-trajectories'

DATASETS = {
    'bhaduri_2020': {
        'accession': 'GSE132672',
        'description': 'Bhaduri et al. 2020 — cortical organoids (3 protocols)',
    },
    'zhong_2018': {
        'accession': 'GSE104276',
        'description': 'Zhong et al. 2018 — fetal PFC atlas, GW8-26',
    },
}

for name, info in DATASETS.items():
    info['raw_dir']       = os.path.join(DRIVE_ROOT, 'data', 'raw', name)
    info['processed_dir'] = os.path.join(DRIVE_ROOT, 'data', 'processed', name)
    os.makedirs(info['raw_dir'], exist_ok=True)
    os.makedirs(info['processed_dir'], exist_ok=True)
    print(f"[{info['accession']}] {info['description']}")
    print(f"  raw:       {info['raw_dir']}")
    print(f"  processed: {info['processed_dir']}\n")

## 3. Verify GEO Accessions — List Available Files

Lists files on GEO FTP for both datasets without downloading anything.  
Confirm the filenames look like count matrices before proceeding.

In [ ]:
import ftplib

def list_geo_suppl(accession):
    prefix = accession[:len(accession)-3] + 'nnn'
    ftp_path = f'/geo/series/{prefix}/{accession}/suppl/'
    ftp = ftplib.FTP('ftp.ncbi.nlm.nih.gov')
    ftp.login()
    try:
        files = ftp.nlst(ftp_path)
        ftp.quit()
        return [f.split('/')[-1] for f in files]
    except ftplib.error_perm as e:
        ftp.quit()
        return [f'ERROR: {e}']

for name, info in DATASETS.items():
    acc = info['accession']
    print(f"--- {acc} ({info['description']}) ---")
    for f in list_geo_suppl(acc):
        print(f'  {f}')
    print()

## 4. Download Both Datasets

Downloads all supplementary files for each dataset to its `raw_dir` on Drive.  
Skips files that already exist (safe to re-run).

In [ ]:
import ftplib
import os

def download_geo_suppl(accession, dest_dir):
    prefix = accession[:len(accession)-3] + 'nnn'
    ftp_path = f'/geo/series/{prefix}/{accession}/suppl/'
    ftp = ftplib.FTP('ftp.ncbi.nlm.nih.gov')
    ftp.login()
    files = ftp.nlst(ftp_path)
    print(f'  {len(files)} file(s) found')
    for filepath in files:
        filename = filepath.split('/')[-1]
        local_path = os.path.join(dest_dir, filename)
        if os.path.exists(local_path):
            size_mb = os.path.getsize(local_path) / 1e6
            print(f'  SKIP (exists, {size_mb:.1f} MB): {filename}')
            continue
        print(f'  Downloading: {filename} ...', end=' ', flush=True)
        with open(local_path, 'wb') as f:
            ftp.retrbinary(f'RETR {filepath}', f.write)
        size_mb = os.path.getsize(local_path) / 1e6
        print(f'done ({size_mb:.1f} MB)')
    ftp.quit()

for name, info in DATASETS.items():
    print(f"\n=== {info['accession']} — {name} ===")
    download_geo_suppl(info['accession'], info['raw_dir'])

print('\nAll downloads complete.')

## 5. Verify Downloads — List Files on Drive

In [ ]:
for name, info in DATASETS.items():
    raw_dir = info['raw_dir']
    print(f"\n--- {name} ({raw_dir}) ---")
    total_mb = 0
    for fname in sorted(os.listdir(raw_dir)):
        fpath = os.path.join(raw_dir, fname)
        size_mb = os.path.getsize(fpath) / 1e6
        total_mb += size_mb
        print(f'  {fname:<60} {size_mb:>8.1f} MB')
    print(f'  Total: {total_mb:.1f} MB')

## 6. Install scanpy

In [ ]:
!pip install -q scanpy

## 7. Sanity Check — Peek at Each Dataset

Load each file and print shape + metadata columns.  
No filtering or modification — just confirming the data loaded correctly.

In [ ]:
import scanpy as sc

def load_dataset(raw_dir, label):
    files = sorted([
        f for f in os.listdir(raw_dir)
        if not f == 'filelist.txt'
    ])
    print(f'\n=== {label} ===')
    for i, f in enumerate(files):
        print(f'  [{i}] {f}')

    # Load the first non-trivial file automatically
    data_file = os.path.join(raw_dir, files[0])
    print(f'\n  Loading: {files[0]} ...')

    if data_file.endswith('.h5ad'):
        adata = sc.read_h5ad(data_file)
    elif data_file.endswith('.h5'):
        adata = sc.read_10x_h5(data_file)
    elif data_file.endswith('.txt.gz') or data_file.endswith('.txt'):
        # GEO text matrices: genes x cells — transpose to cells x genes
        adata = sc.read_text(data_file).T
    else:
        print('  Unrecognized format — load manually.')
        return None

    print(f'  Shape:       {adata.shape[0]:,} cells x {adata.shape[1]:,} genes')
    print(f'  obs columns: {list(adata.obs.columns)}')
    print(f'  var columns: {list(adata.var.columns)}')
    print(f'  First 3 barcodes: {list(adata.obs_names[:3])}')
    return adata

adata_organoid = load_dataset(DATASETS['bhaduri_2020']['raw_dir'], 'Bhaduri 2020 — organoids')
adata_fetal    = load_dataset(DATASETS['zhong_2018']['raw_dir'],   'Zhong 2018 — fetal PFC')

## 8. Save as h5ad

Convert to `.h5ad` format for fast loading in all downstream notebooks.  
No filtering or normalization — this is the raw object.

In [ ]:
def save_h5ad(adata, processed_dir, filename):
    if adata is None:
        print(f'Skipping {filename} — adata is None')
        return
    out_path = os.path.join(processed_dir, filename)
    adata.write_h5ad(out_path)
    size_mb = os.path.getsize(out_path) / 1e6
    print(f'Saved: {out_path} ({size_mb:.1f} MB)')

save_h5ad(adata_organoid, DATASETS['bhaduri_2020']['processed_dir'], 'bhaduri_2020_raw.h5ad')
save_h5ad(adata_fetal,    DATASETS['zhong_2018']['processed_dir'],   'zhong_2018_raw.h5ad')

## Done

Data is now on Google Drive:
- `data/raw/bhaduri_2020/` — original GEO files
- `data/raw/zhong_2018/` — original GEO files
- `data/processed/bhaduri_2020/bhaduri_2020_raw.h5ad`
- `data/processed/zhong_2018/zhong_2018_raw.h5ad`

Next: `colab_01_preprocessing.ipynb` — QC, filtering, normalization, HVG, PCA on both datasets.